In [2]:
!pip install git+https://github.com/facebookresearch/ImageBind.git#egg=imagebind
!pip install setuptools<82
!pip install torchcodec

  Cloning https://github.com/facebookresearch/ImageBind.git to /tmp/pip-install-k9bkqnz0/imagebind_b6703072cb9a4ac68dd3e7b84bd38712
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/ImageBind.git /tmp/pip-install-k9bkqnz0/imagebind_b6703072cb9a4ac68dd3e7b84bd38712
  Resolved https://github.com/facebookresearch/ImageBind.git to commit 53680b02d7e37b19b124fa37bae4b6c98c38f5be
  Preparing metadata (setup.py) ... done
  Cloning https://github.com/facebookresearch/pytorchvideo.git (to revision 6cdc929315aab1b5674b6dcf73b16ec99147735f) to /tmp/pip-install-k9bkqnz0/pytorchvideo_ba992acd0c2b4f33831fe1879d5e59eb
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/pytorchvideo.git /tmp/pip-install-k9bkqnz0/pytorchvideo_ba992acd0c2b4f33831fe1879d5e59eb
  Running command git rev-parse -q --verify 'sha^6cdc929315aab1b5674b6dcf73b16ec99147735f'
  Running command git fetch -q https://github.com/facebookresearch/pytorchvid

In [3]:
import torch
from imagebind.models.imagebind_model import ImageBindModel
from imagebind.models import imagebind_model
from imagebind import data
from imagebind.models.imagebind_model import ModalityType
import polars as pl
import random

/usr/local/lib/python3.13/dist-packages/imagebind/data.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [6]:
manifest_path = "drive/MyDrive/pokemon_data/manifest.jsonl"
device = "cuda:0" # "cpu", "cuda:0"

model = ImageBindModel(
    vision_embed_dim=1280,
    vision_num_blocks=32,
    vision_num_heads=16,
    text_embed_dim=1024,
    text_num_blocks=24,
    text_num_heads=16,
    out_embed_dim=1024,
    audio_drop_path=0.1,
    imu_drop_path=0.7,
)

In [4]:
#torch.cuda.get_device_name(0)

'NVIDIA RTX PRO 6000 Blackwell Server Edition'

In [7]:
model.load_state_dict(torch.load("drive/MyDrive/imagebind/.checkpoints/imagebind_huge.pth", weights_only=True))
model.eval()
model.to(device)

ImageBindModel(
  (modality_preprocessors): ModuleDict(
    (vision): RGBDTPreprocessor(
      (cls_token): tensor((1, 1, 1280), requires_grad=True)

      (rgbt_stem): PatchEmbedGeneric(
        (proj): Sequential(
          (0): PadIm2Video()
          (1): Conv3d(3, 1280, kernel_size=(2, 14, 14), stride=(2, 14, 14), bias=False)
        )
      )
      (pos_embedding_helper): SpatioTemporalPosEmbeddingHelper(
        (pos_embed): tensor((1, 257, 1280), requires_grad=True)

      )
    )
    (text): TextPreprocessor(
      (pos_embed): tensor((1, 77, 1024), requires_grad=True)
      (mask): tensor((77, 77), requires_grad=False)

      (token_embedding): Embedding(49408, 1024)
    )
    (audio): AudioPreprocessor(
      (cls_token): tensor((1, 1, 768), requires_grad=True)

      (rgbt_stem): PatchEmbedGeneric(
        (proj): Conv2d(1, 768, kernel_size=(16, 16), stride=(10, 10), bias=False)
        (norm_layer): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      )
      (pos_e

In [9]:
df = pl.read_ndjson(manifest_path)

use_image_flag = [True if i % 2 == 0 else False for i in range(df.shape[0])]
use_image_flag = random.sample(use_image_flag, len(use_image_flag))
df = df.with_columns(
    use_image=pl.Series(use_image_flag)
)

df

dex_num,name,page_title,image_path,text,use_image
str,str,str,str,str,bool
"""0001""","""Bulbasaur""","""Bulbasaur (Pokémon)""","""data/pokemon_data/images/0001_…","""Bulbasaur is a small, quadrupe…",false
"""0002""","""Ivysaur""","""Ivysaur (Pokémon)""","""data/pokemon_data/images/0002_…","""Ivysaur is a quadrupedal amphi…",false
"""0003""","""Venusaur""","""Venusaur (Pokémon)""","""data/pokemon_data/images/0003_…","""Venusaur is a squat, quadruped…",false
"""0004""","""Charmander""","""Charmander (Pokémon)""","""data/pokemon_data/images/0004_…","""Charmander is a bipedal, repti…",true
"""0005""","""Charmeleon""","""Charmeleon (Pokémon)""","""data/pokemon_data/images/0005_…","""Charmeleon is a bipedal, repti…",false
…,…,…,…,…,…
"""1020""","""Gouging Fire""","""Gouging Fire (Pokémon)""","""data/pokemon_data/images/1020_…","""Gouging Fire is a massive lion…",false
"""1021""","""Raging Bolt""","""Raging Bolt (Pokémon)""","""data/pokemon_data/images/1021_…","""Raging Bolt is a massive quadr…",true
"""1022""","""Iron Boulder""","""Iron Boulder (Pokémon)""","""data/pokemon_data/images/1022_…","""Iron Boulder is a spherical, b…",false


In [10]:
image_df = df.filter(pl.col("use_image") == True)
text_df = df.filter(pl.col("use_image") == False)

images = ["drive/MyDrive/" + path[5:] for path in image_df["image_path"]]
text = text_df["text"]

images[0]

'drive/MyDrive/pokemon_data/images/0004_Charmander.png'

In [13]:
batch_size = 100

round = 0
all_embeddings = []
for i in range(0, len(images), batch_size):
    print("Round", round)
    cur_input = {
        ModalityType.VISION: data.load_and_transform_vision_data(images[i:i+batch_size], device),
        ModalityType.TEXT: data.load_and_transform_text(text[i:i+batch_size], device),
    }
    print("input loaded")

    with torch.amp.autocast(device, dtype=torch.float16):
        with torch.no_grad():
            embeddings = model(cur_input)

    print(embeddings)
    print()
    all_embeddings.append(embeddings)
    round += 1

Round 0
input loaded
{'vision': tensor([[ 0.0076,  0.0307,  0.0468,  ...,  0.0164,  0.0041,  0.0053],
        [ 0.0166,  0.0250,  0.0450,  ...,  0.0098,  0.0100, -0.0019],
        [ 0.0114,  0.0099,  0.0359,  ..., -0.0035, -0.0239,  0.0026],
        ...,
        [ 0.0118,  0.0097,  0.0546,  ...,  0.0083,  0.0128, -0.0010],
        [ 0.0352, -0.0026,  0.0430,  ..., -0.0018, -0.0100, -0.0127],
        [ 0.0046,  0.0251,  0.0486,  ..., -0.0193, -0.0113, -0.0100]],
       device='cuda:0'), 'text': tensor([[ 0.4888,  2.6005,  3.8929,  ...,  0.0635, -1.4891, -0.1580],
        [ 3.1548,  2.0426,  3.4702,  ...,  0.5220,  1.2801,  2.2151],
        [ 3.3492,  3.0044,  1.9368,  ..., -2.0400,  1.0907, -1.3537],
        ...,
        [ 0.9723, -2.9320, -1.8881,  ..., -0.1176,  4.6886,  2.0446],
        [-0.9563,  3.1725,  4.4278,  ..., -0.9422,  3.2412, -1.0328],
        [ 0.7165, -3.6554,  3.1763,  ..., -3.5652,  1.5247, -0.4920]],
       device='cuda:0')}

Round 1
input loaded
{'vision': tensor([[

# Ingesting

In [14]:
!pip install lancedb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.2/75.2 MB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 406.8/406.8 kB 61.3 MB/s eta 0:00:00


In [21]:
all_embeddings[0]['vision'].shape

torch.Size([100, 1024])

In [23]:
all_embeddings[0]['vision'][0:2].shape

torch.Size([2, 1024])

In [16]:
image_df

dex_num,name,page_title,image_path,text,use_image
str,str,str,str,str,bool
"""0004""","""Charmander""","""Charmander (Pokémon)""","""data/pokemon_data/images/0004_…","""Charmander is a bipedal, repti…",true
"""0006""","""Charizard""","""Charizard (Pokémon)""","""data/pokemon_data/images/0006_…","""Charizard is a draconic , bipe…",true
"""0012""","""Butterfree""","""Butterfree (Pokémon)""","""data/pokemon_data/images/0012_…","""Butterfree is a lepidopteran i…",true
"""0013""","""Weedle""","""Weedle (Pokémon)""","""data/pokemon_data/images/0013_…","""Weedle is a larval insect Poké…",true
"""0014""","""Kakuna""","""Kakuna (Pokémon)""","""data/pokemon_data/images/0014_…","""Kakuna is a yellow, insect coc…",true
…,…,…,…,…,…
"""1010""","""Iron Leaves""","""Iron Leaves (Pokémon)""","""data/pokemon_data/images/1010_…","""Iron Leaves is a mechanical Po…",true
"""1012""","""Poltchageist""","""Poltchageist (Pokémon)""","""data/pokemon_data/images/1012_…","""Poltchageist is a Pokémon made…",true
"""1013""","""Sinistcha""","""Sinistcha (Pokémon)""","""data/pokemon_data/images/1013_…","""Sinistcha is a Pokémon made of…",true


In [46]:
text_df

dex_num,name,page_title,image_path,text,use_image
str,str,str,str,str,bool
"""0001""","""Bulbasaur""","""Bulbasaur (Pokémon)""","""data/pokemon_data/images/0001_…","""Bulbasaur is a small, quadrupe…",false
"""0002""","""Ivysaur""","""Ivysaur (Pokémon)""","""data/pokemon_data/images/0002_…","""Ivysaur is a quadrupedal amphi…",false
"""0003""","""Venusaur""","""Venusaur (Pokémon)""","""data/pokemon_data/images/0003_…","""Venusaur is a squat, quadruped…",false
"""0005""","""Charmeleon""","""Charmeleon (Pokémon)""","""data/pokemon_data/images/0005_…","""Charmeleon is a bipedal, repti…",false
"""0007""","""Squirtle""","""Squirtle (Pokémon)""","""data/pokemon_data/images/0007_…","""Squirtle is a small reptilian …",false
…,…,…,…,…,…
"""1019""","""Hydrapple""","""Hydrapple (Pokémon)""","""data/pokemon_data/images/1019_…","""Hydrapple is a Pokémon compose…",false
"""1020""","""Gouging Fire""","""Gouging Fire (Pokémon)""","""data/pokemon_data/images/1020_…","""Gouging Fire is a massive lion…",false
"""1022""","""Iron Boulder""","""Iron Boulder (Pokémon)""","""data/pokemon_data/images/1022_…","""Iron Boulder is a spherical, b…",false


In [43]:
to_ingest = []

round = 0
for i in range(0, len(images), batch_size):
    print("Round", round)

    # vision set
    image_df_part = image_df.slice(i, i+batch_size).to_dict(as_series=False)
    vals = list(image_df_part.values()) + [all_embeddings[round]['vision'].tolist()]
    keys = list(image_df_part.keys()) + ["vector"]

    image_ingest = [dict(zip(keys, values)) for values in zip(*vals)]

    # text set
    text_df_part = text_df.slice(i, i+batch_size).to_dict(as_series=False)
    vals = list(text_df_part.values()) + [all_embeddings[round]['text'].tolist()]
    keys = list(text_df_part.keys()) + ["vector"]

    text_ingest = [dict(zip(keys, values)) for values in zip(*vals)]

    to_ingest += image_ingest + text_ingest
    round += 1

print(len(to_ingest))
to_ingest[0]

Round 0
Round 1
Round 2
Round 3
Round 4
1000


{'dex_num': '0004',
 'name': 'Charmander',
 'page_title': 'Charmander (Pokémon)',
 'image_path': 'data/pokemon_data/images/0004_Charmander.png',
 'text': "Charmander is a bipedal, reptilian Pokémon with a primarily orange body and blue eyes. Its underside from the chest down and the soles of its feet are cream-colored. It has two small fangs visible in its upper jaw and two smaller fangs in its lower jaw. A fire burns at the tip of this Pokémon's slender tail, which has blazed there since Charmander's birth.",
 'use_image': True,
 'vector': [0.0075536202639341354,
  0.030658485367894173,
  0.04675363749265671,
  0.028882469981908798,
  0.016439253464341164,
  0.045199621468782425,
  -0.02256651036441326,
  0.006338159088045359,
  0.0022408333607017994,
  0.05767613649368286,
  -0.015917548909783363,
  -0.007942124269902706,
  0.012543116696178913,
  0.0060051558539271355,
  0.004201389383524656,
  -0.0217451024800539,
  -0.009601589292287827,
  -0.00273340055719018,
  -0.03043648414313

In [63]:
import lancedb

uri = "pokemon_db"
db = lancedb.connect(uri)

In [64]:
table = db.create_table("pokemon", data=to_ingest, mode="overwrite")

In [35]:
len(all_embeddings[0]['text'][0].tolist())

1024

In [70]:
query_vector = all_embeddings[0]['vision'][0].tolist()
#query_vector = all_embeddings[2]['text'][95].tolist()

result = (
    table.search(query_vector)
    .select(["name", "use_image", "_distance"])
    .limit(10)
    .to_polars()
)
print(result)

shape: (10, 3)
┌────────────┬───────────┬───────────┐
│ name       ┆ use_image ┆ _distance │
│ ---        ┆ ---       ┆ ---       │
│ str        ┆ bool      ┆ f32       │
╞════════════╪═══════════╪═══════════╡
│ Charmander ┆ true      ┆ 0.0       │
│ Charizard  ┆ true      ┆ 0.224603  │
│ Dragonite  ┆ true      ┆ 0.246567  │
│ Nidoqueen  ┆ true      ┆ 0.319709  │
│ Eevee      ┆ true      ┆ 0.354788  │
│ Salamence  ┆ true      ┆ 0.367259  │
│ Lickitung  ┆ true      ┆ 0.367272  │
│ Totodile   ┆ true      ┆ 0.371974  │
│ Quagsire   ┆ true      ┆ 0.38685   │
│ Mew        ┆ true      ┆ 0.390053  │
└────────────┴───────────┴───────────┘


In [49]:
#db.drop_table("pokemon")

# Querying

In [16]:
import lancedb

uri = "../../data/pokemon_db"
db = lancedb.connect(uri)

In [17]:
db.list_tables()

ListTablesResponse(context=None, tables=['pokemon'], page_token=None)

In [18]:
table = db.open_table("pokemon")

In [20]:
table.search().to_polars()

dex_num,name,page_title,image_path,text,use_image,vector
str,str,str,str,str,bool,"array[f32, 1024]"
"""0004""","""Charmander""","""Charmander (Pokémon)""","""data/pokemon_data/images/0004_…","""Charmander is a bipedal, repti…",true,"[0.007554, 0.030658, … 0.0053]"
"""0006""","""Charizard""","""Charizard (Pokémon)""","""data/pokemon_data/images/0006_…","""Charizard is a draconic , bipe…",true,"[0.016637, 0.02502, … -0.0019]"
"""0012""","""Butterfree""","""Butterfree (Pokémon)""","""data/pokemon_data/images/0012_…","""Butterfree is a lepidopteran i…",true,"[0.011366, 0.009898, … 0.00258]"
"""0013""","""Weedle""","""Weedle (Pokémon)""","""data/pokemon_data/images/0013_…","""Weedle is a larval insect Poké…",true,"[0.032541, 0.013124, … 0.003622]"
"""0014""","""Kakuna""","""Kakuna (Pokémon)""","""data/pokemon_data/images/0014_…","""Kakuna is a yellow, insect coc…",true,"[0.011231, 0.029003, … 0.000883]"
…,…,…,…,…,…,…
"""1019""","""Hydrapple""","""Hydrapple (Pokémon)""","""data/pokemon_data/images/1019_…","""Hydrapple is a Pokémon compose…",false,"[-1.925704, -0.637682, … 4.399613]"
"""1020""","""Gouging Fire""","""Gouging Fire (Pokémon)""","""data/pokemon_data/images/1020_…","""Gouging Fire is a massive lion…",false,"[-1.295118, 3.737894, … 2.221165]"
"""1022""","""Iron Boulder""","""Iron Boulder (Pokémon)""","""data/pokemon_data/images/1022_…","""Iron Boulder is a spherical, b…",false,"[2.732905, 6.297201, … -0.804262]"


In [ ]:
# Example Query

query_vector = all_embeddings[0]['vision'][0].tolist()
#query_vector = all_embeddings[2]['text'][95].tolist()

result = (
    table.search(query_vector)
    .select(["name", "use_image", "_distance"])
    .limit(10)
    .to_polars()
)
print(result)